# 0. Overview
(Explain notebook's purpose, inputs, and outputs)
## Purpose
This notebook processes the rsults generated by MalSkills scans into structured, readable datasets for further analysis.

MalSkills produces scan summaries and detailed security findings. This notebook extracts and organizes that information into consistent tables, making it easier to examine findings across multiple skill packages and eventually compare MalSkills results with those produced by Cisco.

**Scope**: This notebook is responsible for processing existing scan results only. It does not execute new scans, manually validate findings, or evaluate scanner accuracy.

## Inputs
The notebook reads existing MalSkills output files from a selected scan run:
- `scan_results.jsonl` -- Scan records and package-level results.
- `scan_summary.json` -- Overall scan summary.
- `scan_config.json` -- Configuration used for the scan.
- `cases/` -- Package-specific results, including detailed findings stored in sso_findings.json.

## Outputs
The notebook produces structured files for downstream analysis:
- `package_summary.csv` -- Package-level scan information.
- `security_findings.csv` -- Individual security findings and their associated metadata.
- `behavior_summary.csv` -- Aggregated findings by behaviour category.
- `processing_metadata.json` -- Processing information, including record counts, input validation results, and warnings.

These outputs will provide a foundation for future analysis, reporting, and comparison with Cisco scanner results.

# 1. Configuration
The notebook will need to have access to three locations:
- **Project root**: The main `SkillGuard` folder
- **Input directory**: The folder containing the MalSkills scan results
- **Output directory**: Where the processed CSV and JSON files will be saved.

Run the following cell to complete this step.

In [66]:
from pathlib import Path

# Select project directory
project_root = Path.cwd().parent

# Select the MalSkills scan run to process
run_name = "package_test_1"

# Input and output directories
input_dir = project_root / "results" / "malskills" / run_name
output_dir = input_dir / "processed_v2"

print("Selected run: ", run_name)
print("Input directory: ", input_dir)
print("Directory exists: ", input_dir.is_dir())
print("Output directory: ", output_dir)


Selected run:  package_test_1
Input directory:  c:\Users\ziyoo\Documents\year4\SkillGuard\results\malskills\package_test_1
Directory exists:  True
Output directory:  c:\Users\ziyoo\Documents\year4\SkillGuard\results\malskills\package_test_1\processed_v2


# 2. Load scan records
## Purpose
This section loads the raw scan records generated by MalSkills into a pandas DataFrame, allowing us to inspect and eventually process the results in a tabular format.

Each record contains information about a scanned skill package, such as its identifier, artifact count, and number of reported findings.

Before extracting individual security findings, we first need to understand how the scan results are organized and what information is available.

## Input files
The selected run contains three primary files and a directory:
- `scan_results.jsonl` -- Individual scan records and package-level results
- `scan_summary.json` -- Overall summary of the scan
- `scan_config.json` -- Configuration used during scanning
- `cases/` -- Package-specific files, including detailed security findings

For this section, we start with `scan_results.jsonl`

## Loading the data
The scan records are stored in **JSON Lines (JSONL)** format, where each line contains a separate JSON record.

Using `pd.read_json()` with `lines=True`, we load these records into a pandas DataFrame (`scan_df`), where each row represents a scan record and each column represents an attribute of that record.

## Inspecting the structure

Before processing the results, we inspect the DataFrame to understand the dataset's structure.

- **File existence**: Confirms that the selected scan results file is accessible.
- **First five records (head())**: Provides an initial view of the data and how values are represented.
- **Number of records (len())**: Determines how many scan records were loaded.
- **Column names (columns)**: Identifies the available attributes, such as entry_id, artifact_count, and finding_count.

This inspection helps us determine which fields are relevant to our research, identify what information may require additional extraction, and understand how the scan records relate to the detailed findings stored in `cases/`.

It also establishes a starting point for checking data completeness and consistency in the next section.

**Output**: A DataFrame named scan_df containing the loaded scan records, ready for input verification and subsequent processing.


In [67]:
import pandas as pd

# Load scan_results.jsonl
scan_results_path = input_dir / "scan_results.jsonl"

print("File exists:", scan_results_path.is_file())

# Load scan records and store it in scan_df
scan_df = pd.read_json(scan_results_path, lines=True) # Read the data into a DataFrame and use the JSONL format.

# Display the first five records
display(scan_df.head())

# Inspect structure
print("Number of records:", len(scan_df))
print("Columns:", scan_df.columns.tolist())


File exists: True


,artifact_count,entry_id,error,finding_count,llm_only_finding_count,observed_candidate_count,operand_count,operand_resolution_count,package,package_path,...,retained_output,runtime_sec,scan_path,semgrep_status,skill_content_sha256,skill_file,source,sso_count,status,verdict
0,1,exported_skills::skill_0000,,0,0,0,0,0,skill_0000,exported_skills/skill_0000,...,cases/exported_skills_skill_0000__9cb3b0d275,59.3468,/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/...,ok,9d079a4ba8287b82e2e8f46f3426ca6cdec6d68a35b203...,exported_skills/skill_0000/SKILL.md,exported_skills,0,ok,benign
1,1,exported_skills::skill_0001,,0,0,0,0,0,skill_0001,exported_skills/skill_0001,...,cases/exported_skills_skill_0001__b81182eb53,59.3998,/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/...,ok,25ffa8f05b7ddcf775a669bf5089e662def9af305fce35...,exported_skills/skill_0001/SKILL.md,exported_skills,0,ok,benign
2,1,exported_skills::skill_0003,,0,0,0,0,0,skill_0003,exported_skills/skill_0003,...,cases/exported_skills_skill_0003__c9c1c695a8,59.1168,/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/...,ok,9dffbc3379352bb4e53e3dbd876704fc0ea558a242c658...,exported_skills/skill_0003/SKILL.md,exported_skills,0,ok,benign
3,2,exported_skills::skill_0002,,0,0,0,0,0,skill_0002,exported_skills/skill_0002,...,cases/exported_skills_skill_0002__13a28b8864,59.6094,/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/...,ok,2cdc07f13ab6edc005e8039995381bc92b522798f04e98...,exported_skills/skill_0002/SKILL.md,exported_skills,0,ok,benign
4,1,exported_skills::skill_0005,,0,0,0,0,0,skill_0005,exported_skills/skill_0005,...,cases/exported_skills_skill_0005__7680bc7999,59.5950,/mnt/c/Users/ziyoo/Documents/year4/SkillGuard/...,ok,4d1f8090560056643b244b78da9a46b05b26c0e7c15fe4...,exported_skills/skill_0005/SKILL.md,exported_skills,0,ok,benign


Number of records: 100
Columns: ['artifact_count', 'entry_id', 'error', 'finding_count', 'llm_only_finding_count', 'observed_candidate_count', 'operand_count', 'operand_resolution_count', 'package', 'package_path', 'pattern_matches', 'patterns', 'proposed_rule_count', 'retained_output', 'runtime_sec', 'scan_path', 'semgrep_status', 'skill_content_sha256', 'skill_file', 'source', 'sso_count', 'status', 'verdict']


# 3. Verify inputs
(check required columns and detailed finding files)
## Purpose
This section checks wehther the loaded MalSkills scacn results contain the information required for processing.

Before extracting invidiual security findings, we need to verify that the necessary fields and supporting files are available.

These checks help identify missing data, incomplete scan records, and potential inconsistencies that could affect the reliability of our processed results.

## Verification steps
1. **Required columns**: Check whether `scan_df` contains the fields needed for subsequent processing
2. **Missing values**: Identify missing information in important fields.
3. **Scan errors**: Examine whether any records report errors during scanning.
4. **Detailed findings files**: Verify that the expected `sso_findings.json` files are available for records that reference them.

**Output**: A summary of input verification results, identifying potential issues before extracting and normalizing security findings.

### 3.1 Dataset structure and completeness

In [68]:
# Summarize the dataset structure
print("Records:", len(scan_df))
print("Columns:", len(scan_df.columns))

# Verify required columns
required_columns = [
    "entry_id",
    "package",
    "status",
    "finding_count"
]

missing_columns = [
    col for col in required_columns
    if col not in scan_df.columns
]

print("\nMissing required columns: ", missing_columns)

# Check for missing values in required_columns
print("\nMissing values: ")
print(scan_df[required_columns].isna().sum())


Records: 100
Columns: 23

Missing required columns:  []

Missing values: 
entry_id         0
package          0
status           0
finding_count    0
dtype: int64


### 3.2 Scan execution status

In [69]:
# Count records by scan status
print("\nRecord count: ")
print(scan_df["status"].value_counts(dropna=False))

# Count non-empty error messages
error_count = scan_df["error"].fillna("").str.strip().ne("").sum()

print("\nRecords with reported errors:", error_count)

print("\nSemgrep statuses:")
print(scan_df["semgrep_status"].value_counts(dropna=False))


Record count: 
status
ok    100
Name: count, dtype: int64

Records with reported errors: 0

Semgrep statuses:
semgrep_status
ok    100
Name: count, dtype: int64


### 3.3 Detailed findings availability

In [70]:
# Check the availability of detailed findings files
cases_dir = input_dir / "cases"

# Get all subdirectories inside cases/
case_dirs = [p for p in cases_dir.iterdir() if p.is_dir()]

# Find sso_findings.json in each case directory
finding_files = [
    p / "sso_findings.json"
    for p in case_dirs
    if (p / "sso_findings.json").is_file()
]

# Compare num of cases with available findings files
print("Case directories:", len(case_dirs))
print("Finding files:", len(finding_files))

Case directories: 100
Finding files: 100


# 4. Extract findings
**Goal**: Convert the security findings stored in individual JSON files into a structured dataset that we can analyze with pandas.

### 4.1 Inspect a sample findings file
Prior to extracting security findings, we inspect the structure of `sso_findings.json` to determine how the findings are organized and which attributes are available.

This allows us to identify the fields needed for extraction and determine how to convert the JSON data into a pandas DataFrame.

In [71]:
import json

# Select a sample findings file to inspect its structure
sample_file = finding_files[0]

# Open the JSON file and load its contents into a Python object
with open(sample_file, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

# Display the filename and the type of data stored in it
print("File: ", sample_file.name)
print("Data type: ", type(sample_data).__name__)

# If the JSON contains a dictionary, display its available keys
if isinstance(sample_data, dict):
    print("Keys: ", list(sample_data.keys()))

# If the JSON contains a list, inspect its size and first record
elif isinstance(sample_data, list):
    print("Number of items: ", len(sample_data))
    if sample_data:
        print("First item: ", sample_data[0])

File:  sso_findings.json
Data type:  list
Number of items:  0


### 4.2 Inspect the fields of a finding
After confirming that `sso_findings.json` contains a list of records, we inspect the structure of an individual finding.

This helps use identify which attributes are available, determine whether any fields contain nested data, and decide how to organize the findings into a DataFrame.

The following output should provide you with three groups of information: 
- **Identification**: Which artifact and finding we're observing
- **Classification**: How the finding is classified and what text was matched
- **Detection details**: Information about the detection process, supporting metadata, and location

In [72]:
import json

# Find a findings file containing at least one detection
sample_file = None
sample_data = None

for file in finding_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list) and len(data) > 0:
        sample_file = file
        sample_data = data
        break

if sample_file is not None:
    print("File:", sample_file.name)
    print("Number of findings:", len(sample_data))
else:
    print("No nonempty findings files found.")


File: sso_findings.json
Number of findings: 16


### 4.3 Inspect a finding's values
Before building the DataFrame, we shoudl inspect the actual values of a few fields. This will help us understand what MalSkills is detecting and how it represents those detectinos.

In [73]:
# Inspect the classification and detection details
fields = [
    "category",
    "subtype",
    "matched_text",
    "producer"
]

for field in fields:
    print(f"{field}: {sample_finding.get(field)}")

# Inspect selected attributes
print("\nDetection attributes:")

attributes = sample_finding["attributes"]

for field in ["engine", "rule_id", "sink_api", "sso_category"]:
    print(f"{field}: {attributes.get(field)}")


category: network_access
subtype: connection_create
matched_text: - SSH key configuration: HIGH — contains authentication credentials
producer: semgrep

Detection attributes:
engine: semgrep
rule_id: mnt.c.Users.ziyoo.Documents.year4.MalSkills.malskills.rules.semgrep.markdown.markdown.network_and_remote_communication.connection_command
sink_api: 
sso_category: network_access


### 4.4 Extract findings from all case files

After inspecting the structure and values of an individual finding, we extract the findings from all available `sso_findings.json` files.

Each file contains a list of finding records associated with a scanned case. We combine these records into a single dataset while preserving their source case.

Nested dictionaries are flattened into separate columns to make the findings easier to filter, compare, and analyze.

In [74]:
# Store findings from all case files
all_findings = []

# Read each available findings file
for file_path in finding_files:
    with open(file_path, "r", encoding="utf-8") as f:
        findings = json.load(f)

    # Preserve the case directory associated with each finding
    for finding in findings:
        finding["case_id"] = file_path.parent.name
        all_findings.append(finding)

# Convert nested JSON records into a DataFrame
findings_df = pd.json_normalize(all_findings)

# Inspect the extracted dataset
print("Total findings:", len(findings_df))
print("Columns:", len(findings_df.columns))

display(findings_df)
# display(findings_df.head(choose_num)) # If you want to see a specific amount


Total findings: 391
Columns: 34


,artifact_id,artifact_path,category,finding_id,matched_text,producer,subtype,case_id,attributes.analysis_component,attributes.analysis_stage,...,provenance.analysis_component,provenance.analysis_stage,provenance.artifact.id,provenance.artifact.path,provenance.producer,provenance.span.end_line,provenance.span.start_line,span.end_line,span.start_line,attributes.resource_class
0,art_0000,DEPENDENCIES.md,software_installation,semgrep_00000,,semgrep,package_installation,exported_skills_skill_0004__a768cf0aa4,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0000,DEPENDENCIES.md,semgrep,24,24,24,24,NaN
1,art_0000,DEPENDENCIES.md,software_installation,semgrep_00001,- **Miniforge/Miniconda** - 环境管理（推荐）,semgrep,package_installation,exported_skills_skill_0004__a768cf0aa4,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0000,DEPENDENCIES.md,semgrep,25,25,25,25,NaN
2,art_0000,DEPENDENCIES.md,software_installation,semgrep_00002,,semgrep,package_installation,exported_skills_skill_0004__a768cf0aa4,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0000,DEPENDENCIES.md,semgrep,26,26,26,26,NaN
3,art_0000,DEPENDENCIES.md,software_installation,semgrep_00003,- **Git** - 用于版本控制,semgrep,package_installation,exported_skills_skill_0004__a768cf0aa4,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0000,DEPENDENCIES.md,semgrep,27,27,27,27,NaN
4,art_0001,README.md,software_installation,semgrep_00004,- Home Assistant 已安装并运行,semgrep,package_installation,exported_skills_skill_0004__a768cf0aa4,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0001,README.md,semgrep,31,31,31,31,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,art_0017,scripts/deploy-execute.mjs,payload_execution,semgrep_00024,process.exit(1);,semgrep,system_command_execution,exported_skills_skill_0097__e5af1b358c,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0017,scripts/deploy-execute.mjs,semgrep,503,503,503,503,NaN
387,art_0017,scripts/deploy-execute.mjs,payload_execution,semgrep_00025,process.exit(0);,semgrep,system_command_execution,exported_skills_skill_0097__e5af1b358c,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0017,scripts/deploy-execute.mjs,semgrep,508,508,508,508,NaN
388,art_0018,scripts/deploy-preflight.mjs,payload_execution,semgrep_00026,"return execSync(cmd, { encoding: ""utf8"", stdio...",semgrep,system_command_execution,exported_skills_skill_0097__e5af1b358c,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0018,scripts/deploy-preflight.mjs,semgrep,28,28,28,28,NaN
389,art_0018,scripts/deploy-preflight.mjs,payload_execution,semgrep_00027,process.exit(1);,semgrep,system_command_execution,exported_skills_skill_0097__e5af1b358c,semgrep_finding,sso_extraction,...,semgrep_finding,sso_extraction,art_0018,scripts/deploy-preflight.mjs,semgrep,51,51,51,51,NaN


### 4.5 Verify extraction

In [75]:
# Compare reported findings with extracted findings
reported_count = scan_df["finding_count"].sum()
extracted_count = len(findings_df)

print("Reported findings:", reported_count)
print("Extracted findings:", extracted_count)
print("Counts match:", reported_count == extracted_count)


Reported findings: 391
Extracted findings: 391
Counts match: True


# 5. Normalize data
The findings extracted in Section 4 contain information about detected security-sensitive behaviors, including nested attributes and provenance information.

Before summarizing the results, we inspect the extracted fields to identify overlapping information, check for missing or inconsistent values, and determine which columns are relevant to our analysis.

This section focuses on:

1. **Field inspection:** Compare related columns to understand their contents and identify potential redundancies.
2. **Data quality:** Check for missing values, empty strings, and inconsistencies.
3. **Standardization:** Create a separate DataFrame with consistent missing-value representations while preserving the original extracted findings.

The resulting dataset will be used to summarize the security-sensitive behaviors reported by MalSkills.

### 5.1 Inspect extracted columns

In [76]:
# Display all extracted columns
print(findings_df.columns.tolist())

['artifact_id', 'artifact_path', 'category', 'finding_id', 'matched_text', 'producer', 'subtype', 'case_id', 'attributes.analysis_component', 'attributes.analysis_stage', 'attributes.engine', 'attributes.engines', 'attributes.message', 'attributes.producer', 'attributes.producers', 'attributes.rule_id', 'attributes.rule_ids', 'attributes.rule_origin', 'attributes.rule_origins', 'attributes.ruleset_digest', 'attributes.schema_version', 'attributes.sink_api', 'attributes.source_rule_category', 'attributes.sso_category', 'provenance.analysis_component', 'provenance.analysis_stage', 'provenance.artifact.id', 'provenance.artifact.path', 'provenance.producer', 'provenance.span.end_line', 'provenance.span.start_line', 'span.end_line', 'span.start_line', 'attributes.resource_class']


### 5.2 Compare detection sources

In [77]:
# Compare the different fields identifying the detection source
producer_columns = [
    "producer",
    "attributes.producer",
    "provenance.producer",
    "attributes.engine"
]

display(findings_df[producer_columns])

,producer,attributes.producer,provenance.producer,attributes.engine
0,semgrep,semgrep,semgrep,semgrep
1,semgrep,semgrep,semgrep,semgrep
2,semgrep,semgrep,semgrep,semgrep
3,semgrep,semgrep,semgrep,semgrep
4,semgrep,semgrep,semgrep,semgrep
...,...,...,...,...
386,semgrep,semgrep,semgrep,semgrep
387,semgrep,semgrep,semgrep,semgrep
388,semgrep,semgrep,semgrep,semgrep
389,semgrep,semgrep,semgrep,semgrep


### 5.3 Inspect behavior classifications

In [78]:
# Compare fields describing detected security behaviors
behavior_columns = [
    "category",
    "subtype",
    "attributes.sso_category",
    "attributes.source_rule_category"
]

display(findings_df[behavior_columns])

,category,subtype,attributes.sso_category,attributes.source_rule_category
0,software_installation,package_installation,software_installation,
1,software_installation,package_installation,software_installation,
2,software_installation,package_installation,software_installation,
3,software_installation,package_installation,software_installation,
4,software_installation,package_installation,software_installation,
...,...,...,...,...
386,payload_execution,system_command_execution,payload_execution,Command Execution
387,payload_execution,system_command_execution,payload_execution,Command Execution
388,payload_execution,system_command_execution,payload_execution,Command Execution
389,payload_execution,system_command_execution,payload_execution,Command Execution


The extracted findings contain multiple fields describing security-sensitive behaviours. In the current sample, `category` and `attributes.sso_category` contain identical values, while `subtype` provides a more specific classification. The `attributes.source_rule_category` field provides additional rule information but is not populated for every finding.

### 5.4. Inspect artifact identifiers

In [79]:
# Compare fields identifying the artifact associated with each finding
artifact_columns = [
    "artifact_id",
    "artifact_path",
    "provenance.artifact.id",
    "provenance.artifact.path",
    "case_id"
]

display(findings_df[artifact_columns])

,artifact_id,artifact_path,provenance.artifact.id,provenance.artifact.path,case_id
0,art_0000,DEPENDENCIES.md,art_0000,DEPENDENCIES.md,exported_skills_skill_0004__a768cf0aa4
1,art_0000,DEPENDENCIES.md,art_0000,DEPENDENCIES.md,exported_skills_skill_0004__a768cf0aa4
2,art_0000,DEPENDENCIES.md,art_0000,DEPENDENCIES.md,exported_skills_skill_0004__a768cf0aa4
3,art_0000,DEPENDENCIES.md,art_0000,DEPENDENCIES.md,exported_skills_skill_0004__a768cf0aa4
4,art_0001,README.md,art_0001,README.md,exported_skills_skill_0004__a768cf0aa4
...,...,...,...,...,...
386,art_0017,scripts/deploy-execute.mjs,art_0017,scripts/deploy-execute.mjs,exported_skills_skill_0097__e5af1b358c
387,art_0017,scripts/deploy-execute.mjs,art_0017,scripts/deploy-execute.mjs,exported_skills_skill_0097__e5af1b358c
388,art_0018,scripts/deploy-preflight.mjs,art_0018,scripts/deploy-preflight.mjs,exported_skills_skill_0097__e5af1b358c
389,art_0018,scripts/deploy-preflight.mjs,art_0018,scripts/deploy-preflight.mjs,exported_skills_skill_0097__e5af1b358c


The extracted findings contain multiple fields identifying the artifacts associated with detected behaviors.

In the current sample, `artifact_id` and `provenance.artifact.id` contain identical values. Similarly, `artifact_path` and `provenance.artifact.path` match across all findings.

The `case_id` field identifies the scanned case associated with each finding. Multiple findings may share the same case and artifact identifiers.

These observations help distinguish individual detections from the artifacts and cases associated with them when summarizing results.

### 5.5 Inspect missing values

Check the extracted findings for missing values and empty strings before normalizing the dataset.

In [80]:
# Count missing values across all extracted columns
missing_values = findings_df.isna().sum()

# Display only columns containing missing values
print(missing_values[missing_values > 0])

# Count empty or whitespace-only strings in each column
empty_strings = findings_df.apply(
    lambda col: col.map(
        lambda value: isinstance(value, str) and value.strip() == ""
    ).sum()
)

# Display columns containing empty strings
print(empty_strings[empty_strings > 0])

attributes.resource_class    388
dtype: int64
matched_text                         7
attributes.sink_api                286
attributes.source_rule_category    286
dtype: int64


In [81]:
# Display findings with empty sink_api or source_rule_category
columns = [
    "finding_id",
    "category",
    "subtype",
    "attributes.sink_api",
    "attributes.source_rule_category"
]

display(
    findings_df[
        (findings_df["attributes.sink_api"].str.strip() == "") |
        (findings_df["attributes.source_rule_category"].str.strip() == "")
    ][columns]
)

,finding_id,category,subtype,attributes.sink_api,attributes.source_rule_category
0,semgrep_00000,software_installation,package_installation,,
1,semgrep_00001,software_installation,package_installation,,
2,semgrep_00002,software_installation,package_installation,,
3,semgrep_00003,software_installation,package_installation,,
4,semgrep_00004,software_installation,package_installation,,
...,...,...,...,...,...
358,semgrep_00022,system_configuration,system_configuration_modify,,
359,semgrep_00023,file_operation,file_read,,
360,semgrep_00025,network_access,data_receive,,
361,semgrep_00026,network_access,data_receive,,


The extracted findings contain missing and empty attribute values. The `attributes.resource_class` field contains 388 missing values, while `matched_text`, `attributes.sink_api`, and `attributes.source_rule_category `contain 7, 286, and 286 empty strings, respectively.

These values will be standardized during normalization to ensure consistent representation of missing data. No findings will be removed solely because these attributes are missing.

### 5.6 Standardize missing values

Convert empty or whitespace-only strings into consistent missing-value representations while preserving the original extracted findings.

In [82]:
# Create a separate copy to preserve the original findings
normalized_df = findings_df.copy()

# Convert empty or whitespace-only strings into missing values
normalized_df = normalized_df.replace(r"^\s*$", pd.NA, regex=True)

# Verify the updated missing-value counts
print(normalized_df.isna().sum()[lambda counts: counts > 0])

matched_text                         7
attributes.sink_api                286
attributes.source_rule_category    286
attributes.resource_class          388
dtype: int64


A separate DataFrame, `normalized_df`, was created to preserve the original extracted findings.

Empty and whitespace-only strings were converted to `pd.NA` to ensure consistent representation of missing values.

After normalization, `attributes.sink_api` and `attributes.source_rule_category` each contain one missing value. No findings were removed.

The normalization process standardized empty and whitespace-only strings into missing-value markers (`pd.NA`). A comparison between the original and normalized DataFrames confirmed that 579 values were converted across three columns: `matched_text` (7), `attributes.sink_api` (286), and `attributes.source_rule_category` (286).

The normalized dataset retains all 391 findings and 34 columns, preserving the original records while ensuring consistent representation of missing values.

# 6. Summarize results
Aggregate the normalized MalSkills findings to examine the distribution of detected security-sensitive behaviors across the scanned skills.

This section summarizes the number of findings, affected skills, affected artifacts, and behavior classifications.

### 6.1 Count findings and affected skills

In [83]:
# Count the total number of detected findings
total_findings = len(normalized_df)

# Count unique scanned cases containing findings
affected_skills = normalized_df["case_id"].nunique()

print("Total findings:", total_findings)
print("Skills with findings:", affected_skills)

Total findings: 391
Skills with findings: 29


The initial MalSkills analysis produced 391 security-sensitive findings across 29 distinct scanned skills.

This distinction is important because a single skill may produce multiple findings. Therefore, the total number of findings does not necessarily correspond to the number of affected skills.

Of the 100 skills scanned, 29% contained at least one detected security-sensitive behavior. However, these findings do not necessarily indicate that the affected skills are malicious or vulnerable.

### 6.2 Summarize behavior categories

In [84]:
# Count findings belonging to each security behavior category
category_counts = normalized_df["category"].value_counts()

display(category_counts)

category
file_operation           129
sensitive_data_access     96
network_access            73
payload_execution         48
software_installation     35
cryptography               7
system_configuration       2
process_operation          1
Name: count, dtype: int64

# 7. Export results
(Save standardized tables and processing metadata)

### 7.1 Configure the export directory

In [85]:

from pathlib import Path

# Save processed results in a separate directory
export_dir = Path("../results/processed")
export_dir.mkdir(parents=True, exist_ok=True)

print("Export directory:", export_dir.resolve())

Export directory: C:\Users\ziyoo\Documents\year4\SkillGuard\results\processed


### 7.2 Export the processed findings

In [94]:
import json

# Create a separate copy for CSV export
csv_df = normalized_df.copy()

# Identify columns containing lists
list_columns = [
    col for col in csv_df.columns
    if csv_df[col].map(lambda value: isinstance(value, list)).any()
]

# Serialize lists into JSON strings
for col in list_columns:
    csv_df[col] = csv_df[col].map(
        lambda value: json.dumps(value)
        if isinstance(value, list) else value
    )

# Export the processed findings
output_file = export_dir / "security_findings.csv"
csv_df.to_csv(output_file, index=False)

print("Exported:", output_file)
print("Serialized list columns:", list_columns)

Exported: ..\results\processed\security_findings.csv
Serialized list columns: ['attributes.engines', 'attributes.producers', 'attributes.rule_ids', 'attributes.rule_origins']


# 8. Processing verification
(Check totals, missing records, and export integrity)

### 8.1 Verify exported findings
Reload the exported CSV file and compare its dimensions with the normalized DataFrame to verify that all findings and columns were preserved during export.

In [95]:
# Reload the exported findings
exported_df = pd.read_csv(output_file)

# Compare the original and exported dimensions
print("Normalized DataFrame:", normalized_df.shape)
print("Exported DataFrame:", exported_df.shape)

# Check whether the dimensions match
print("Dimensions match:", normalized_df.shape == exported_df.shape)

Normalized DataFrame: (391, 34)
Exported DataFrame: (391, 34)
Dimensions match: True


### 8.2 Verify exported data integrity
Compare the normalized DataFrame with the reloaded CSV to verify that the exported data preserves the original values and missing-value representations.

In [96]:
from pandas.testing import assert_frame_equal

# Reconstruct the serialized lists
restored_df = exported_df.copy()

for col in list_columns:
    restored_df[col] = restored_df[col].map(
        lambda value: json.loads(value)
        if isinstance(value, str) else value
    )

# Verify that the restored data matches
assert_frame_equal(
    normalized_df,
    restored_df,
    check_dtype=False
)

print("Exported data matches normalized data.")

Exported data matches normalized data.
